# Pass 3A — Publication-Grade Baseline Audit

**Breast Ultrasound Classification (BUSI / VGG16)**

This notebook audits the dataset and reproducibility conditions before the publication-grade VGG16 baseline is rerun.

### Objectives
1. Inventory the BUSI images and class distribution.
2. Audit the filename-based identifier.
3. Detect exact duplicate images.
4. Check image readability and dimensions.
5. Audit lesion-mask availability and integrity.
6. Optionally verify raw-ROI/preprocessed-ROI correspondence.
7. Save a master manifest for the subsequent leakage-controlled split.

> **Important:** `filename_number` is deliberately *not* called `patient_id`. Its meaning must be independently established before claiming a patient-level split.


## 1. Configuration

In [ ]:
from pathlib import Path
import os
import random
import hashlib
import re
import json

import numpy as np
import pandas as pd
import cv2

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# EDIT THESE PATHS IF NEEDED
DATA_ROOT = Path("../data/Dataset_BUSI_with_GT")
RAW_ROI_ROOT = Path("../data/raw_roi_dataset")
PREP_ROI_ROOT = Path("../data/preprocessed_roi_dataset")

OUTPUT_DIR = Path("../results/publication_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["benign", "malignant", "normal"]
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg"}

print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 2. Verify the dataset root

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {DATA_ROOT.resolve()}\n"
        "Update DATA_ROOT in the configuration cell."
    )

for cls in CLASSES:
    class_dir = DATA_ROOT / cls
    print(f"{cls:10s} -> {'OK' if class_dir.exists() else 'MISSING'}")


## 3. Inventory original BUSI images

In [ ]:
def list_images(root: Path):
    records = []
    for class_name in CLASSES:
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for path in sorted(class_dir.iterdir()):
            if (
                path.is_file()
                and path.suffix.lower() in SUPPORTED_EXTENSIONS
                and "_mask" not in path.stem.lower()
            ):
                records.append(path)
    return records

image_paths = list_images(DATA_ROOT)
print("Total original images:", len(image_paths))


In [ ]:
manifest = pd.DataFrame([
    {
        "image_path": str(path),
        "filename": path.name,
        "class": path.parent.name,
        "filename_number": None,
    }
    for path in image_paths
])

print("\nClass distribution:")
print(manifest["class"].value_counts().sort_index())


## 4. Audit the filename-based identifier

In [ ]:
def extract_filename_number(path: Path):
    match = re.search(r"\((\d+)\)", path.stem)
    return int(match.group(1)) if match else None

manifest["filename_number"] = [
    extract_filename_number(Path(p))
    for p in manifest["image_path"]
]

print("Missing filename numbers:", manifest["filename_number"].isna().sum())
print("\nUnique filename numbers by class:")
print(manifest.groupby("class")["filename_number"].nunique(dropna=True))


### 4.1 Identifier collision audit

A repeated number does **not** prove repeated patients, and a unique number does **not** prove patient identity. This section is diagnostic only.

In [ ]:
identifier_audit = (
    manifest
    .groupby("filename_number", dropna=False)
    .agg(
        n_images=("image_path", "count"),
        n_classes=("class", "nunique"),
        classes=("class", lambda x: sorted(set(x))),
        filenames=("filename", list),
    )
    .reset_index()
    .sort_values(["n_classes", "n_images"], ascending=[False, False])
)

print("Identifier groups:", len(identifier_audit))
display(identifier_audit.head(30))

cross_class_collisions = identifier_audit[identifier_audit["n_classes"] > 1].copy()
same_identifier_multiple_images = identifier_audit[identifier_audit["n_images"] > 1].copy()

print("\nFilename numbers occurring across multiple classes:", len(cross_class_collisions))
print("Filename numbers associated with >1 image:", len(same_identifier_multiple_images))
if len(cross_class_collisions):
    display(cross_class_collisions.head(20))


### Interpretation rule

Do **not** rename `filename_number` to `patient_id` based only on this audit. Verify the BUSI naming convention/source documentation first.

## 5. Exact duplicate-image audit

In [ ]:
def md5_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest["md5"] = [md5_file(Path(p)) for p in manifest["image_path"]]

duplicate_groups = (
    manifest.groupby("md5").size()
    .reset_index(name="n_images")
    .query("n_images > 1")
    .sort_values("n_images", ascending=False)
)

print("Exact duplicate hash groups:", len(duplicate_groups))
print(
    "Images contained in duplicate groups:",
    int(duplicate_groups["n_images"].sum()) if len(duplicate_groups) else 0
)
display(duplicate_groups.head(20))


In [ ]:
if len(duplicate_groups):
    exact_duplicates = (
        manifest[manifest["md5"].isin(duplicate_groups["md5"])]
        .sort_values("md5")
    )
    display(exact_duplicates[["image_path", "class", "filename", "md5"]])
else:
    print("No exact duplicate image files detected.")


## 6. Image readability and dimensions

In [ ]:
def image_properties(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return {"readable": False, "height": None, "width": None, "dtype": None}
    h, w = image.shape
    return {"readable": True, "height": h, "width": w, "dtype": str(image.dtype)}

properties_df = pd.DataFrame([image_properties(Path(p)) for p in manifest["image_path"]])
manifest = pd.concat([manifest.reset_index(drop=True), properties_df], axis=1)

print("Unreadable images:", int((~manifest["readable"]).sum()))
print("\nImage dimensions:")
display(
    manifest.groupby(["height", "width"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)


## 7. Lesion-mask integrity audit

In [ ]:
def expected_mask_path(image_path: Path):
    return image_path.with_name(f"{image_path.stem}_mask{image_path.suffix}")

manifest["mask_path"] = [str(expected_mask_path(Path(p))) for p in manifest["image_path"]]
manifest["mask_exists"] = [Path(p).exists() for p in manifest["mask_path"]]

print("Mask availability by class:")
display(
    manifest.groupby("class")["mask_exists"]
    .value_counts()
    .rename("count")
    .reset_index()
)


In [ ]:
def mask_properties(mask_path):
    path = Path(mask_path)
    if not path.exists():
        return {"mask_readable": False, "mask_nonzero_pixels": None, "mask_shape": None}
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return {"mask_readable": False, "mask_nonzero_pixels": None, "mask_shape": None}
    return {
        "mask_readable": True,
        "mask_nonzero_pixels": int(np.count_nonzero(mask)),
        "mask_shape": tuple(mask.shape),
    }

mask_info_df = pd.DataFrame([mask_properties(p) for p in manifest["mask_path"]])
manifest = pd.concat([manifest.reset_index(drop=True), mask_info_df], axis=1)

print(
    "Unreadable existing masks:",
    int((~manifest["mask_readable"] & manifest["mask_exists"]).sum())
)
print(
    "Existing masks with zero non-zero pixels:",
    int((
        manifest["mask_exists"]
        & manifest["mask_readable"]
        & (manifest["mask_nonzero_pixels"] == 0)
    ).sum())
)


## 8. Optional ROI correspondence audit

Checks whether raw-ROI and preprocessed-ROI directories contain matching filenames. Edit the paths in the configuration cell if necessary.

In [ ]:
def roi_files(root):
    root = Path(root)
    if not root.exists():
        return pd.DataFrame(columns=["class", "filename", "path"])

    records = []
    for cls in ["benign", "malignant"]:
        class_dir = root / cls
        if not class_dir.exists():
            continue
        for path in sorted(class_dir.iterdir()):
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
                records.append({"class": cls, "filename": path.name, "path": str(path)})
    return pd.DataFrame(records)

raw_df = roi_files(RAW_ROI_ROOT)
prep_df = roi_files(PREP_ROI_ROOT)

print("Raw ROI files:", len(raw_df))
print("Preprocessed ROI files:", len(prep_df))

if len(raw_df) and len(prep_df):
    comparison = raw_df.merge(
        prep_df,
        on=["class", "filename"],
        how="outer",
        indicator=True,
        suffixes=("_raw", "_prep")
    )
    print("\nCorrespondence:")
    print(comparison["_merge"].value_counts())
    mismatches = comparison[comparison["_merge"] != "both"]
    if len(mismatches):
        display(mismatches.head(50))
    else:
        print("All raw and preprocessed ROI filenames correspond.")
else:
    print("ROI directories were not both found; set their paths if needed.")


## 9. Generate and save the master manifest

In [ ]:
manifest["audit_status"] = np.where(manifest["readable"], "OK", "UNREADABLE_IMAGE")

master_columns = [
    "image_path", "filename", "class", "filename_number", "md5",
    "height", "width", "dtype", "mask_path", "mask_exists",
    "mask_readable", "mask_nonzero_pixels", "audit_status"
]

master_manifest = manifest[master_columns].copy()
master_path = OUTPUT_DIR / "master_manifest_pre_split.csv"
master_manifest.to_csv(master_path, index=False)

print("Saved:", master_path.resolve())
print("Rows:", len(master_manifest))


## 10. Save an audit summary

In [ ]:
summary = {
    "seed": SEED,
    "dataset_root": str(DATA_ROOT.resolve()),
    "n_images": int(len(manifest)),
    "class_counts": {k: int(v) for k, v in manifest["class"].value_counts().to_dict().items()},
    "unreadable_images": int((~manifest["readable"]).sum()),
    "missing_filename_numbers": int(manifest["filename_number"].isna().sum()),
    "duplicate_hash_groups": int(len(duplicate_groups)),
    "images_in_duplicate_groups": int(
        duplicate_groups["n_images"].sum() if len(duplicate_groups) else 0
    ),
    "missing_masks": int((~manifest["mask_exists"]).sum()),
    "unreadable_existing_masks": int(
        (~manifest["mask_readable"] & manifest["mask_exists"]).sum()
    ),
}

summary_path = OUTPUT_DIR / "audit_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path.resolve())


# Pass 3A — Stop Point

This notebook intentionally stops **before model training and before assigning train/test labels**.

### Required checks
- [ ] Dataset inventory completed
- [ ] Filename identifier audited
- [ ] Exact duplicates checked
- [ ] Images readable
- [ ] Masks checked
- [ ] ROI correspondence checked if applicable
- [ ] Meaning of the filename identifier independently established

**Do not create `patient_id` by simply renaming `filename_number`.**

Next: **Pass 3B — lock the leakage-controlled master split and rerun the VGG16 baseline under the same split across whole-image, raw-ROI, and preprocessed-ROI conditions.**
